# 基于 ResNet 的 K 线图涨跌趋势分类研究\n## 魔搭社区 (ModelScope) GPU 实验\n\n### 运行前准备\n1. `git clone https://github.com/Heartunderbladde/CV_Restnet.git`\n2. 将 `all_ohlc_data.pkl` 上传到 `data/raw/` 目录下\n3. 按顺序运行每个 Cell\n\n**环境**: Ubuntu 22.04 | CUDA 12.8 | PyTorch 2.10 | NVIDIA A10 24GB

## Cell 0: 环境检查 + 路径设置

In [ ]:
import os, sys\n\n# 自动检测当前 notebook 所在目录作为 BASE_DIR\nBASE_DIR = os.getcwd()\nif not os.path.exists(os.path.join(BASE_DIR, 'src', 'config.py')):\n    # 如果不在项目根目录，尝试常见位置\n    for guess in ['/mnt/workspace/CV_Restnet', '/mnt/workspace']:\n        if os.path.exists(os.path.join(guess, 'src', 'config.py')):\n            BASE_DIR = guess\n            os.chdir(BASE_DIR)\n            break\n\nos.environ['MODELSCOPE_BASE_DIR'] = BASE_DIR\nsys.path.insert(0, os.path.join(BASE_DIR, 'src'))\n\nprint(f'BASE_DIR: {BASE_DIR}')\nprint(f'src: {os.path.join(BASE_DIR, "src")}')\nprint()\n\n# 检查 GPU\nimport torch\nprint(f'PyTorch: {torch.__version__}')\nprint(f'CUDA available: {torch.cuda.is_available()}')\nif torch.cuda.is_available():\n    print(f'GPU: {torch.cuda.get_device_name(0)}')\n    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3\n    print(f'VRAM: {mem_gb:.0f} GB')\n\n# 检查 src 目录\nprint()\n!ls {BASE_DIR}/src/

## Cell 1: 安装依赖

In [ ]:
!pip install mplfinance scikit-learn tqdm akshare -i https://mirrors.aliyun.com/pypi/simple/ --trusted-host mirrors.aliyun.com -q\nprint('依赖安装完成')

## Cell 2: 加载数据 (Phase 1)\n先运行此 Cell 创建目录，再上传 `all_ohlc_data.pkl` 到打印的路径

In [ ]:
from config import *\n\ndata_path = os.path.join(RAW_DATA_DIR, 'all_ohlc_data.pkl')\nprint(f'数据路径: {data_path}')\nprint()\n\nif os.path.exists(data_path):\n    import pickle\n    with open(data_path, 'rb') as f:\n        all_data = pickle.load(f)\n    print(f'数据加载成功: {len(all_data)} 只股票')\n    \n    total_samples = sum(len(df) for df in all_data.values())\n    print(f'总样本数: {total_samples}')\n    \n    sample_code = list(all_data.keys())[0]\n    print(f'列名: {all_data[sample_code].columns.tolist()}')\nelse:\n    print(f'数据文件不存在!')\n    print(f'请将 all_ohlc_data.pkl 上传到: {RAW_DATA_DIR}/')\n    print(f'上传后重新运行此 Cell')

## Cell 3: 渲染 K 线图 (Phase 2)\nSTRIDE=15, 约 26k 张图, 多进程渲染

In [ ]:
%%time\n%run {BASE_DIR}/src/02_render_charts.py

## Cell 4: 验证 DataLoader (Phase 3)

In [ ]:
from dataset import get_dataloaders\n\ntrain_loader, val_loader, test_loader = get_dataloaders()\nprint(f'Train batches: {len(train_loader)}')\nprint(f'Val batches: {len(val_loader)}')\nprint(f'Test batches: {len(test_loader)}')\n\nimgs, labels = next(iter(train_loader))\nprint(f'Input shape: {imgs.shape}')\nprint(f'Label shape: {labels.shape}')\nprint(f'Label dist: up={(labels==1).sum().item()}, down={(labels==0).sum().item()}')

## Cell 5: 构建模型 (Phase 4)

In [ ]:
from model import build_model\n\nmodel = build_model()\nprint(f'Model on {DEVICE}')\n\nmem = sum(p.numel() * p.element_size() for p in model.parameters()) / 1024**2\nprint(f'Params memory: {mem:.1f} MB')\nprint(f'Estimated GPU usage: ~{mem*3 + 500:.0f} MB (batch=128)')

## Cell 6: 训练 (Phase 5)\nGPU 训练, 预计 15-20 分钟

In [ ]:
%%time\n%run {BASE_DIR}/src/train.py

## Cell 7: 测试集评估 (Phase 6)

In [ ]:
%run {BASE_DIR}/src/evaluate.py

## Cell 8: 结果展示

In [ ]:
from IPython.display import Image, display\n\nresults_path = f'{OUTPUT_DIR}/results.txt'\nif os.path.exists(results_path):\n    with open(results_path, 'r', encoding='utf-8') as f:\n        print(f.read())\n\nfor name in ['confusion_matrix.png', 'roc_curve.png', 'training_curves.png']:\n    path = f'{OUTPUT_DIR}/{name}'\n    if os.path.exists(path):\n        print(f'--- {name} ---')\n        display(Image(filename=path))\n        print()

## (可选) 查看所有输出

In [ ]:
print('=== 输出文件 ===')\nfor root, dirs, files in os.walk(OUTPUT_DIR):\n    for f in files:\n        path = os.path.join(root, f)\n        size_mb = os.path.getsize(path) / 1024**2\n        print(f'{size_mb:5.1f} MB  {path}')\n\nprint()\nprint('=== 图片统计 ===')\nfor split in ['train', 'val', 'test']:\n    for label in ['up', 'down']:\n        d = os.path.join(IMAGE_DIR, split, label)\n        if os.path.exists(d):\n            cnt = len([f for f in os.listdir(d) if f.endswith('.png')])\n            print(f'  {split}/{label}: {cnt}')